# Boosting Trees (solucion)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, RocCurveDisplay
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import load_breast_cancer


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_NOTEBOOKS = "https://github.com/AdanReyes2407/Notebooks_NM_ML.git"


def is_colab():
    return "google.colab" in sys.modules


def ensure_repo_clone(repo_url=REPO_NOTEBOOKS, target=Path('/content/Notebooks_NM_ML')):
    if is_colab() and not target.exists():
        print(f"Clonando repositorio en {target} ...")
        subprocess.run(["git", "clone", repo_url, str(target)], check=True)


def resolve_data_dir():
    ensure_repo_clone()
    candidates = [
        Path('/content/Notebooks_NM_ML/Bases de datos'),
        Path.cwd() / 'Bases de datos',
        Path.cwd().parent / 'Bases de datos',
        Path('/content/Bases de datos'),
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


In [ ]:
def load_dataset(data_dir):
    if data_dir is not None:
        p = data_dir / 'spambase.data'
        if p.exists():
            df = pd.read_csv(p, header=None)
            X = df.iloc[:, :-1].copy()
            y = df.iloc[:, -1].astype(int).copy()
            return X, y, 'spambase.data'

    bc = load_breast_cancer(as_frame=True)
    X = bc.data.copy()
    y = bc.target.astype(int).copy()
    return X, y, 'sklearn_breast_cancer'


DATA_DIR = resolve_data_dir()
X, y, fuente = load_dataset(DATA_DIR)
print('Fuente:', fuente)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=11, stratify=y
)

gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=2,
    random_state=11,
)
gb.fit(X_train, y_train)

y_hat = gb.predict(X_test)
y_prob = gb.predict_proba(X_test)[:, 1]

print('Accuracy:', round(accuracy_score(y_test, y_hat), 4))
print('AUC:', round(roc_auc_score(y_test, y_prob), 4))


In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 4.2))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax)
ax.set_title('Curva ROC - Gradient Boosting')
plt.tight_layout()
plt.show()


In [ ]:
rows = []
for lr in [0.01, 0.03, 0.05, 0.1]:
    for n_est in [120, 250, 400, 600]:
        mdl = GradientBoostingClassifier(
            n_estimators=n_est,
            learning_rate=lr,
            max_depth=2,
            random_state=11,
        )
        mdl.fit(X_train, y_train)
        auc = roc_auc_score(y_test, mdl.predict_proba(X_test)[:, 1])
        acc = accuracy_score(y_test, mdl.predict(X_test))
        rows.append((lr, n_est, acc, auc))

res = pd.DataFrame(rows, columns=['learning_rate', 'n_estimators', 'accuracy', 'auc'])
res = res.sort_values('auc', ascending=False)
print(res.head(10))
